# About
This notebook is a step by step cell run to build out the full C3S dataset from raw json sets into a .db file.

What you need are:
- src_data folder: where all the json files live
- C3SDB_schema : for sqlite schema of features
- mqn_schema : for sqlite schema of MQNs
- pred_CCS_scema : for sqlite schema of predicted CCS
- build_utils : folder for extra functions to build out the db file

# Load all the functions

In [2]:
import sqlite3
import os
import pandas as pd 
import requests

# from c3sdb.build_utils.db_init import create_db
# from c3sdb.build_utils.src_data import add_dataset
# from c3sdb.build_utils.smiles import (
#     load_smiles_search_cache,
#     save_smiles_search_cache,
#     add_smiles_to_db,
# )
# from c3sdb.build_utils.mqns import add_mqns_to_db
# from c3sdb.build_utils.classification import label_class_byname
# from c3sdb.build_utils.clean_src import clean_database


In [3]:
conn = sqlite3.connect('C3S_normal.db')
df_normal = pd.read_sql_query("SELECT * FROM master", conn)
conn.close()

In [4]:
conn = sqlite3.connect('C3S_amogh.db')
df_amogh = pd.read_sql_query("SELECT * FROM master", conn)
conn.close()

In [5]:
print(f"The size of C3S_normal is {df_normal.shape[0]}")
print(f"The size of C3S_amogh is {df_amogh.shape[0]}")

The size of C3S_normal is 17679
The size of C3S_amogh is 19581


In [ ]:
amogh_src_tags = df_amogh['src_tag']
normal_src_tags = df_normal['src_tag']

#building a frequency table of src_tags in C3S_amogh
src_tag_freq_amogh = {}
for tag in amogh_src_tags:
    if tag in src_tag_freq_amogh:
        src_tag_freq_amogh[tag] += 1
    else:
        src_tag_freq_amogh[tag] = 1


src_tag_freq_normal = {}
for tag in normal_src_tags:
    if tag in src_tag_freq_normal:
        src_tag_freq_normal[tag] += 1
    else:
        src_tag_freq_normal[tag] = 1

The src_tags that are in C3S_amoght but not in C3S_normal


In [21]:
print(sorted(src_tag_freq_amogh.items()))
print(sorted(src_tag_freq_normal.items()))

for tag_a in src_tag_freq_amogh:
    for tag_b in src_tag_freq_normal:
        if tag_a == tag_b and src_tag_freq_amogh[tag_a] != src_tag_freq_normal[tag_b]:
            print(f"{tag_a} in amogh db has: {src_tag_freq_amogh[tag_a]}")
            print(f"{tag_b} in normal db has: {src_tag_freq_normal[tag_b]}")
            print(f"The difference is: {src_tag_freq_amogh[tag_a] - src_tag_freq_normal[tag_b]}")
            print("--------------------------------")


[('baker0524', 2387), ('belo0321', 311), ('bijl0517', 205), ('blaz0818', 429), ('celm1120', 970), ('dodd0220', 48), ('groe0815', 131), ('hine0119', 179), ('hine0217', 257), ('hine0817', 1426), ('hine1217', 163), ('leap0219', 405), ('lian0118', 126), ('may_0114', 498), ('moll0218', 357), ('mull_1223', 187), ('nich1118', 1078), ('pagl0314', 96), ('palm_0424', 18), ('pola0620', 336), ('righ0218', 106), ('ross0422', 4412), ('stow0817', 86), ('teja0918', 173), ('tsug0220', 2950), ('zhen0917', 949), ('zhou0817', 451), ('zhou1016', 847)]
[('baker0524', 2387), ('belo0321', 311), ('bijl0517', 205), ('blaz0818', 429), ('celm1120', 970), ('dodd0220', 48), ('groe0815', 131), ('hine0119', 179), ('hine0217', 257), ('hine0817', 1426), ('hine1217', 163), ('leap0219', 405), ('lian0118', 126), ('may_0114', 498), ('moll0218', 357), ('mull_1223', 187), ('nich1118', 1078), ('pagl0314', 96), ('palm_0424', 18), ('pola0620', 336), ('righ0218', 106), ('ross0422', 2510), ('stow0817', 86), ('teja0918', 173), ('t

In [33]:
#investigating ross0422 file

import json 
with open('src_data/ross0422.json', 'r') as f:
    data = json.load(f)
    data = data['data']

print(len(data))

# Create a set of tuples for each entry in the original ross0422.json for easy comparison
original_rows = set()
for row in data:
    # Use a tuple of (name, mz, ccs, adduct) as the unique identifier
    original_rows.add((
        row.get('name'),
        row.get('mz'),
        row.get('ccs'),
        row.get('adduct')
    ))

# Filter amogh dataframe for rows with src_tag == 'ross0422'
amogh_ross0422 = df_amogh[df_amogh['src_tag'] == 'ross0422']

# Create a set of tuples for each entry in the amogh dataframe with src_tag 'ross0422'
amogh_rows = set()
for _, row in amogh_ross0422.iterrows():
    amogh_rows.add((
        row.get('name'),
        row.get('mz'),
        row.get('ccs'),
        row.get('adduct')
    ))

# Find rows in amogh that are not in the original json
extra_in_amogh = amogh_rows - original_rows

print(f"Number of entries in original ross0422.json: {len(original_rows)}")
print(f"Number of entries in amogh db with src_tag 'ross0422': {len(amogh_rows)}")
print(f"Number of entries in amogh but not in original: {len(extra_in_amogh)}")
print("Entries in amogh but not in original:")

# Create DataFrame for CSV export
extra = pd.DataFrame(columns=["name", "mz", "ccs", "adduct"])
for entry in extra_in_amogh:
    # Convert tuple to dictionary with proper column mapping
    entry_dict = {
        "name": entry[0],
        "mz": entry[1], 
        "ccs": entry[2],
        "adduct": entry[3]
    }
    extra = pd.concat([extra, pd.DataFrame([entry_dict])], ignore_index=True)

extra.to_csv("extra_ross0422.csv")

# Read the existing JSON file
with open("src_data/extra_ross0422.json", "r") as f:
    existing_data = json.load(f)

# Convert tuples to dictionaries and append to the data array
for entry in extra_in_amogh:
    entry_dict = {
        "name": entry[0],
        "mz": entry[1], 
        "ccs": entry[2],
        "adduct": entry[3]
    }
    existing_data["data"].append(entry_dict)

# Write the updated JSON back to the file
with open("src_data/extra_ross0422.json", "w") as f:
    json.dump(existing_data, f, indent=4)


2510
Number of entries in original ross0422.json: 2510
Number of entries in amogh db with src_tag 'ross0422': 4412
Number of entries in amogh but not in original: 2590
Entries in amogh but not in original:
